# **Problema de Agente Viajero (TSP)**

- Emilio Alejandro González Huerta A01286440
- Laurie Camila Hernández Pacheco A01286569
- Ángel Everardo Rodríguez Guevara A01286630
- Alejandro Israel Manducano Rojo A00841759
- Enrique Alexander Luna Sánchez A00574602

---

En la figura se muestra un pequeño problema del agente viajero con siete ciudades. La
ciudad 1 es el lugar de residencia del agente. Por lo tanto, si comienza desde su ciudad, el
agente debe elegir una ruta para visitar cada una de las otras ciudades exactamente una vez
antes de regresar a su punto de partida. Determina el orden óptimo en que debe de visitar
cada una de las ciudades.

In [1]:
aristas = [
    (1, 2, 12), (1, 3, 10), (1, 7, 12),
    (2, 3, 8),  (2, 4, 12),
    (3, 4, 11), (3, 5, 3),  (3, 7, 9),
    (4, 5, 11), (4, 6, 10),
    (5, 6, 6),  (5, 7, 7),
    (6, 7, 9),
]

> ### **Solución mediante Heurístico**

In [ ]:
n = 7
INICIO = 1

# Adjacent list para costos
vecinos = {i: {} for i in range(1, n + 1)}
for i, j, w in aristas:
    vecinos[i][j] = w
    vecinos[j][i] = w
 
pasos = 0

# Vamos a pensarlo como un recursivo greedy
def buscar(u, ruta, costo):
    global pasos

    # Si ya se visitaron todos, vamonos a casa, caso base de éxito
    if len(ruta) == n and 1 in vecinos[u]: # debe existir el retorno a 1
        pasos += 1
        print(f"Regreso: {u} -> 1")
        return ruta + [1], costo + vecinos[u][1]
    
    # Sección greedy del algoritmo que manda a llamar la siguiente ruta más barata
    # del nodo al que avanzamos
    caminos = sorted([v for v in vecinos[u] if v not in ruta],
                      key=lambda v: (vecinos[u][v], v))
    
    for v in caminos:
        pasos += 1
        print(f"Nodo {u} -> {v}")
        resultado = buscar(v, ruta + [v], costo + vecinos[u][v])

        if resultado:
            return resultado
        
        pasos += 1
        print(f"Retrocede: {v} -> {u}")

    # Caso donde no encuentra salida al nodo que llegamos, caso base fallido del recursivo
    return None
 
ruta, costo = buscar(1, [1], 0)
print("\nRuta:", ruta)
print("Costo:", costo)
print("Pasos:", pasos)

Nodo 1 -> 3
Nodo 3 -> 5
Nodo 5 -> 6
Nodo 6 -> 7
Retrocede: 7 -> 6
Nodo 6 -> 4
Nodo 4 -> 2
Retrocede: 2 -> 4
Retrocede: 4 -> 6
Retrocede: 6 -> 5
Nodo 5 -> 7
Nodo 7 -> 6
Nodo 6 -> 4
Nodo 4 -> 2
Regreso: 2 -> 1

Ruta: [1, 3, 5, 7, 6, 4, 2, 1]
Costo: 63
Pasos: 15


> ### **Solución Óptima**

In [1]:
#Librerías
import pulp

In [ ]:
# Grafo no dirigido
c = {}
for i, j, w in aristas:
    c[(i, j)] = w
    c[(j, i)] = w

ciudades = range(1, 8)
n = len(ciudades)
arcos = list(c.keys())

model = pulp.LpProblem("TSP_7_ciudades", pulp.LpMinimize)

x = {(i, j): pulp.LpVariable(f"x_{i}_{j}", cat="Binary") for (i, j) in arcos}
u = {i: pulp.LpVariable(f"u_{i}", lowBound=1, upBound=n - 1, cat="Continuous") for i in ciudades}

# Función objetivo: z = sum(cij * xij)
model += pulp.lpSum(c[a] * x[a] for a in arcos)

# Restricciones
for i in ciudades:
    model += pulp.lpSum(x[(i, j)] for j in ciudades if (i, j) in c) == 1
    model += pulp.lpSum(x[(j, i)] for j in ciudades if (j, i) in c) == 1

# Reestricción de subtours (MTZ), regresa a ciudad 1
for (i, j) in arcos:
    if i != 1 and j != 1:
        model += u[i] - u[j] + n * x[(i, j)] <= n - 1

model.solve(pulp.PULP_CBC_CMD(msg=False))

print("Estado:", pulp.LpStatus[model.status])
print("Costo total óptimo:", pulp.value(model.objective))

# Reconstruir la ruta empezando en la ciudad 1
sig = {i: j for (i, j) in arcos if x[(i, j)].value() > 0.5}
ruta = [1]
while True:
    ruta.append(sig[ruta[-1]])
    if ruta[-1] == 1:
        break
print("Ruta:", " -> ".join(map(str, ruta)))

Estado: Optimal
Costo total óptimo: 63.0
Ruta: 1 -> 2 -> 4 -> 6 -> 7 -> 5 -> 3 -> 1


Análisis completo dentro del reporte

### Fuentes de consulta

PuLP. (s. f.). orloge: OR logs parser. PuLP 4.0.0 documentation. https://coin-or.github.io/pulp/plugins/orloge.html